# 学習済みbenchmark Baselineとの比較

別で学習したS2UTをBaselineとして登録し、その保存設定・preparedデータ・辞書を再利用してalignmentモデルを新規学習します。Baselineの重みを転移モデルへロードしません。
保存snapshotを別ディレクトリへ検証復元し、Baselineと転移モデルを同じnative fairseq生成器・vocoder・評価器で比較します。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import json, os, subprocess, sys, time
from pathlib import Path
REPO_URL = 'https://github.com/Yaaamashiro/alignment-transfer-s2ut.git'
REPO_REVISION = 'd239f7201ceb1dfdaf530490493427e6ea932f39'
BENCHMARK = '/content/drive/MyDrive/ja-en-direct-s2st-benchmark-data/training/experiment-001' #@param {type:"string"}
RUN_NAME = 's2ut-training-experiment-001-paper-fisher-v1' #@param {type:"string"}
PERSISTENT = '/content/drive/MyDrive/alignment-transfer-s2ut-data/benchmark-match-drive-safe-001' #@param {type:"string"}
ROOT = '/content/alignment-transfer-work/benchmark-match-drive-safe-001' #@param {type:"string"}
PREPARE_AUDIO_PACKS = False #@param {type:"boolean"}
RUN_TRANSFER = False #@param {type:"boolean"}
RUN_MONOTONIC = False #@param {type:"boolean"}
GENERATE_WAVEFORMS = False #@param {type:"boolean"}
ASR_BLEU = False #@param {type:"boolean"}
T2TT_UPDATES = 20000 #@param {type:"integer"}
ANNEAL_STEPS = 200000 #@param {type:"integer"}
PYTHON = '/content/alignment-venv/bin/python'
REPO = Path('/content/alignment-transfer-s2ut')
LOGS = Path(ROOT) / 'execution'
PREPARED = Path(BENCHMARK) / 'data/s2ut/fairseq'
BACKUP = Path(BENCHMARK) / 'checkpoints' / RUN_NAME
RESTORED = Path(ROOT) / 'baseline-snapshot'
BASELINE = Path(ROOT) / 'registrations/baseline.json'
MANIFEST = Path(ROOT) / 'manifest/manifest.jsonl'
os.environ['HF_HOME'] = '/content/huggingface'
os.environ['PYTHONUTF8'] = '1'
os.environ['MPLCONFIGDIR'] = '/content/matplotlib-cache'
assert not Path(ROOT).resolve().is_relative_to(Path(BENCHMARK).resolve())
assert not ASR_BLEU or GENERATE_WAVEFORMS

BENCHMARK_RUNTIME_REVISION = '4361e55a99c7f4ea3a00c5c138f6820c34a07d06'
BENCHMARK_RUNTIME = Path('/content/alignment-benchmark-runtime')
os.environ['S2ST_DRIVE_SAFE'] = '1'
os.environ['S2ST_BACKUP_MIN_SECONDS'] = '600'
os.environ['S2ST_BACKUP_MAX_GB'] = '1024'
assert not str(Path(ROOT).resolve()).startswith('/content/drive/')
assert not Path(ROOT).resolve().is_relative_to(Path(PERSISTENT).resolve())


## 固定環境の構築

Python 3.10 / PyTorch 2.7.1 / 公式fairseq固定revisionを構築します。


In [ ]:
if not (REPO / '.git').is_dir():
    subprocess.run(['git', 'clone', '--no-checkout', REPO_URL, str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', REPO_REVISION], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', REPO_REVISION], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv'], check=True)
subprocess.run(['uv', 'venv', '--python', '3.10.22', '--seed', '--allow-existing', '/content/alignment-venv'], check=True)
for packages in [
    ['pip==23.3.2', 'setuptools<81', 'wheel'],
    ['torch==2.7.1', 'torchaudio==2.7.1', 'numpy==1.26.4', 'Cython==3.2.9'],
    ['--no-build-isolation', '-r', 'requirements-fairseq.txt'],
    ['-r', 'requirements.txt'],
]:
    subprocess.run([PYTHON, '-m', 'pip', 'install', *packages], check=True)
if not (BENCHMARK_RUNTIME / '.git').is_dir():
    subprocess.run(['git', 'clone', '--no-checkout', 'https://github.com/Yaaamashiro/ja-en-direct-s2st-benchmark.git', str(BENCHMARK_RUNTIME)], check=True)
subprocess.run(['git', '-C', str(BENCHMARK_RUNTIME), 'fetch', 'origin', BENCHMARK_RUNTIME_REVISION], check=True)
subprocess.run(['git', '-C', str(BENCHMARK_RUNTIME), 'checkout', '--detach', BENCHMARK_RUNTIME_REVISION], check=True)
subprocess.run([PYTHON, '-m', 'pip', 'install', 'PyYAML==6.0.2', '--no-deps', '-e', str(BENCHMARK_RUNTIME)], check=True)
subprocess.run([PYTHON, '-m', 'pip', 'check'], check=True)

def run(name, module, *arguments, gpu=False, env=None):
    arguments = list(map(str, arguments))
    payload = [PYTHON, '-m', module, *arguments]
    # Read-only environment checks and asset downloads have no stage output.
    if '--output' in arguments and module != 'alignment_transfer_s2ut.benchmark' or (
            module == 'alignment_transfer_s2ut.benchmark' and arguments[0] != 'plan'):
        if '--output' in arguments:
            output = Path(arguments[arguments.index('--output') + 1])
            work = output.parent if output.suffix in ('.json', '.jsonl', '.png') else output
            wrapper = [PYTHON, '-m', 'alignment_transfer_s2ut.colab_runtime', 'run',
                '--work', str(work), '--backup', str(Path(PERSISTENT) / 'stages' / name)]
            if module == 'alignment_transfer_s2ut.train_t2tt' or module == 'alignment_transfer_s2ut.benchmark' and arguments[0] == 'train':
                wrapper.append('--training')
            payload = [*wrapper, '--', *payload]
    command = [PYTHON, '-m', 'alignment_transfer_s2ut.pilot', 'run', '--name', name, '--logs', str(LOGS)]
    if gpu:
        command.append('--monitor-gpu')
    return subprocess.run([*command, '--', *payload], check=True, env=env)

if not PREPARE_AUDIO_PACKS:
    run('runtime', 'alignment_transfer_s2ut.pilot', 'runtime')
if not PREPARE_AUDIO_PACKS:
    runtime = json.loads(subprocess.check_output([PYTHON, '-m', 'alignment_transfer_s2ut.pilot', 'runtime'], text=True))
    (Path(ROOT) / 'runtime.json').write_text(json.dumps(runtime, indent=2), encoding='utf-8')
    print(runtime)
run('cpu-tests', 'pytest', '-q', '-m', 'not gpu')


## native FP16接続試験

小規模な人工データで公式TrainerのFP16更新とpriorのFP32保持を確認します。GPUランタイムで実行してください。実CTC/HuBERT/vocoderを含む統合試験とは別です。


In [ ]:
if not PREPARE_AUDIO_PACKS:
    subprocess.run([PYTHON, '-m', 'pytest', 'tests/test_benchmark_native.py::test_native_fp16_trainer_and_fp32_prior',
        '-q', '--basetemp', '/content/native-fp16-test'], cwd=REPO, check=True)


## benchmarkのDrive I/O・再開処理を使用

作業・checkpoint・ログ・pair artifactは`ROOT`（Colabローカル）へ保存し、Driveの`PERSISTENT`へZIP snapshotを保存します。通常の再開も同じ設定で上から実行します。旧Drive直接保存runとは別の出力先を使用します。
入力はbenchmarkの既存WAV ZIPを読み、原本の論理pathと学習条件を保ったままローカルへ展開します。missing packの場合はbenchmarkと同じCPU音声ZIP準備を使用します。`PREPARE_AUDIO_PACKS=True`でこのセルまでCPUで実行し、GPUへ切り替えた後はFalseへ戻してください。GPU処理中に原本からZIPを作りません。
学習はbenchmarkの常駐process・600秒間隔の単一upload worker・検証付きZIP・完了marker・容量予算を共用します。alignment固有のpair artifactだけ、完了済みファイルの増分ZIPを保存します。未完了コピーを完成扱いせず、既存snapshotを自動削除しません。


In [ ]:
stage_command = [PYTHON, '-m', 'alignment_transfer_s2ut.colab_runtime', 'stage-inputs',
    '--benchmark', BENCHMARK, '--local', '/content/alignment-transfer-inputs/benchmark-match-drive-safe-001']
if PREPARE_AUDIO_PACKS:
    subprocess.run([*stage_command, '--create-audio'], check=True)
    raise RuntimeError('CPU音声ZIP準備完了。GPUへ切り替え、PREPARE_AUDIO_PACKS=Falseで上から再実行してください。')
staged = json.loads(subprocess.check_output(stage_command, text=True))
os.environ['S2ST_INPUT_MAP'] = staged['input_map']
print(staged)


## 既存Baselineの復元と登録

Driveで確認した保存形式は`snapshot.json`と`snapshot.zip`です。初回は最新snapshotを選び、以後の再開では同じsnapshotを維持します。新しいBaselineで比較するときは新しいROOTを指定します。元のbackup/preparedデータは変更しません。


In [ ]:
run('restore-baseline', 'alignment_transfer_s2ut.benchmark', 'restore', '--backup', BACKUP, '--output', RESTORED, '--resume')
run('register-baseline', 'alignment_transfer_s2ut.benchmark', 'register',
    '--checkpoint', RESTORED / 'checkpoints/checkpoint_best.pt', '--data', PREPARED,
    '--recipe-lock', RESTORED / 's2ut-recipe-lock.json', '--metadata', RESTORED / 'research-metadata.json',
    '--output', BASELINE, '--resume')
baseline = json.loads(BASELINE.read_text())
print({key: baseline[key] for key in ['checkpoint_updates', 'training_budget', 'checkpoint_selection', 'selection_policy']})
assert 1 <= ANNEAL_STEPS < baseline['training_budget']

BACKBONE_CONFIG = Path(ROOT) / 'backbone.json'
BACKBONE_CONFIG.write_text(json.dumps({'backbone': baseline['backbone']}), encoding='utf-8')


## 同一のtrain/devとunitを使用

少量pilotへ抽出せず、Baselineが使った全train/devを読み込みます。testはteacher作成・モデル選択に使いません。HuBERT unitはbenchmarkの保存済み列をimportし、prepared targetと一致することを確認します。


In [ ]:
run('full-manifest', 'alignment_transfer_s2ut.benchmark_data', 'manifest',
    '--common', Path(BENCHMARK) / 'data/common', '--prepared', PREPARED, '--output', MANIFEST, '--resume')
run('import-units', 'alignment_transfer_s2ut.benchmark_data', 'units', '--manifest', MANIFEST,
    '--benchmark-units', Path(BENCHMARK) / 'data/s2ut/units', '--prepared', PREPARED,
    '--output', Path(ROOT) / 'units', '--resume')
rows = [json.loads(line) for line in MANIFEST.read_text().splitlines() if line.strip()]


## T2TT学習・BLEU・alignment準備

翻訳teacherは同じ全train pairから学習し、dev text BLEUとalignment診断を確認します。T2TTの学習はS2UT Baselineの継続学習とは別です。

`RUN_TRANSFER=True`でAnnealed T2TT Guidanceを学習します。`RUN_MONOTONIC=True`かつ`RUN_TRANSFER=False`の場合、このteacher準備セルはスキップします。両方Falseの既定では従来どおり主比較のteacher準備とplan確認を行います。


In [ ]:
# Preserve primary preparation by default; a Monotonic-only run needs no teacher stages.
if RUN_TRANSFER or not RUN_MONOTONIC:
    run('t2tt', 'alignment_transfer_s2ut.train_t2tt', '--manifest', MANIFEST,
        '--config', BACKBONE_CONFIG, '--output', Path(ROOT) / 't2tt', '--max-updates', T2TT_UPDATES,
        '--device', 'cuda', '--loader-workers', 2, '--validation-interval', 1000, '--save-interval-updates', 100, '--resume-if-exists', gpu=True)
    run('t2tt-dev', 'alignment_transfer_s2ut.evaluate_t2tt', '--manifest', MANIFEST,
        '--checkpoint', Path(ROOT) / 't2tt/checkpoint_best.pt', '--output', Path(ROOT) / 'text-evaluation/dev-text.json',
        '--device', 'cuda', '--resume', gpu=True)
    print(json.loads((Path(ROOT) / 'text-evaluation/dev-text.json').read_text()))
    run('attention', 'alignment_transfer_s2ut.extract_t2tt_attention', '--manifest', MANIFEST,
        '--checkpoint', Path(ROOT) / 't2tt/checkpoint_best.pt', '--output', Path(ROOT) / 'mt', '--device', 'cuda', '--resume', gpu=True)
    run('ctc', 'alignment_transfer_s2ut.align_speech_text', '--manifest', MANIFEST,
        '--config', 'configs/alignment/ctc.yaml', '--output', Path(ROOT) / 'ctc', '--device', 'cuda', '--resume', gpu=True)
    run('teacher', 'alignment_transfer_s2ut.build_teacher_alignment', '--manifest', MANIFEST,
        '--mt-attention', Path(ROOT) / 'mt', '--alignment', Path(ROOT) / 'ctc', '--units', Path(ROOT) / 'units',
        '--config', BACKBONE_CONFIG, '--output', Path(ROOT) / 'teacher', '--resume')
    run('diagnostics', 'alignment_transfer_s2ut.diagnose_alignment', '--manifest', MANIFEST,
        '--teacher', Path(ROOT) / 'teacher', '--units', Path(ROOT) / 'units', '--config', BACKBONE_CONFIG,
        '--split', 'dev', '--max-samples', 500, '--heatmaps', 20, '--output', Path(ROOT) / 'audit', '--overwrite')


## 保存済みBaselineと同じ条件で転移モデルを新規学習

保存済みcfgから公式fairseq Trainer・FP16・token-budget batch・optimizer・scheduler・補助lossを構築します。更新数はBaselineの完了済み学習budgetに合わせます。report-only unit-loss選択runでは同じ間隔のdev unit lossでbestを選択します。planは学習を開始しません。

任意のcontrolは`RUN_MONOTONIC=True`で実行するAnnealed Monotonic Guidanceです。別の`annealed-monotonic` directoryを使い、`--prior-type monotonic`を指定します。teacherは不要で、Baseline・manifest・units・annealing終了はT2TT側と同じです。


In [ ]:
TRANSFER = Path(ROOT) / 'annealed-t2tt'
TRANSFER_ARGS = ['--baseline', str(BASELINE), '--output', str(TRANSFER), '--method', 'annealed_guidance',
    '--manifest', str(MANIFEST), '--units', str(Path(ROOT) / 'units'), '--teacher', str(Path(ROOT) / 'teacher'),
    '--anneal-steps', str(ANNEAL_STEPS), '--resume']
MONOTONIC = Path(ROOT) / 'annealed-monotonic'
MONOTONIC_ARGS = ['--baseline', str(BASELINE), '--output', str(MONOTONIC), '--method', 'annealed_guidance',
    '--prior-type', 'monotonic', '--manifest', str(MANIFEST), '--units', str(Path(ROOT) / 'units'),
    '--anneal-steps', str(ANNEAL_STEPS), '--resume']

if RUN_MONOTONIC:
    monotonic_plan = json.loads(subprocess.check_output([PYTHON, '-m', 'alignment_transfer_s2ut.benchmark', 'plan', *MONOTONIC_ARGS], text=True))
    print({'condition': 'Annealed Monotonic Guidance', 'training_budget': monotonic_plan['training_budget'],
           'initialization': monotonic_plan['initialization'], 'selection': monotonic_plan['selection_policy']})
    run('native-monotonic', 'alignment_transfer_s2ut.benchmark', 'train', *MONOTONIC_ARGS, gpu=True)

# Keep the existing T2TT command (default prior_type=t2tt) stable for resume.
if RUN_TRANSFER or not RUN_MONOTONIC:
    plan = json.loads(subprocess.check_output([PYTHON, '-m', 'alignment_transfer_s2ut.benchmark', 'plan', *TRANSFER_ARGS], text=True))
    print({'training_budget': plan['training_budget'], 'initialization': plan['initialization'], 'selection': plan['selection_policy']})
    if RUN_TRANSFER:
        run('native-transfer', 'alignment_transfer_s2ut.benchmark', 'train', *TRANSFER_ARGS, gpu=True)


## 共通のteacher-free推論とdev評価

Baselineを再学習せず、全条件で同じnative fairseq beam 10 / max_len_a 1 / max_len_b 200を使います。転移モデルも推論時はstock S2UTとして読み込み、teacherを必要としません。ASR-BLEUは両モデル共通の設定です。

主比較はS2UT BaselineとAnnealed T2TT Guidanceです。`RUN_MONOTONIC=True`の場合はAnnealed Monotonic Guidanceも同じ経路に追加します。Monotonicだけを新規学習した場合はBaselineとの2条件を評価でき、完成済みT2TTがあれば3条件を比較します。指定した条件が未完成なら比較を開始しません。

比較レポートは条件の組み合わせごとに保存し、各条件の生成・評価cacheは共用します。表のUERは比率、ASR-BLEU未実行は「—」です。音声ASR-BLEUを測る場合は冒頭から`GENERATE_WAVEFORMS=True`・`ASR_BLEU=True`で実行してください。


In [ ]:
from IPython.display import Markdown, display

TRANSFER_RECORD = Path(ROOT) / 'registrations/annealed-t2tt.json'
MONOTONIC_RECORD = Path(ROOT) / 'registrations/annealed-monotonic.json'
transfer_conditions = []
if RUN_MONOTONIC:
    transfer_conditions.append(('annealed-monotonic', 'Annealed Monotonic Guidance', MONOTONIC, MONOTONIC_RECORD, 'register-monotonic'))
# Reuse a completed T2TT run even when its training switch is off.
if RUN_TRANSFER or not RUN_MONOTONIC or (TRANSFER / 'checkpoints/checkpoint_best.pt').is_file():
    transfer_conditions.append(('annealed-t2tt', 'Annealed T2TT Guidance', TRANSFER, TRANSFER_RECORD, 'register-transfer'))
missing = [label for _, label, folder, _, _ in transfer_conditions
           if not (folder / 'checkpoints/checkpoint_best.pt').is_file()]
if missing:
    print('学習完了後に再実行してください: ' + ', '.join(missing))
else:
    conditions = [('baseline', 'S2UT Baseline', BASELINE)]
    for name, label, folder, record, registration in transfer_conditions:
        run(registration, 'alignment_transfer_s2ut.benchmark', 'register-transfer', '--run', folder,
            '--checkpoint', folder / 'checkpoints/checkpoint_best.pt', '--output', record, '--resume')
        conditions.append((name, label, record))
    records = [record for _, _, record in conditions]
    comparison_name = '-'.join(name for name, _, _ in conditions)
    COMPARISON_DIR = Path(ROOT) / 'comparison' / comparison_name
    run(comparison_name + '-compare-training', 'alignment_transfer_s2ut.benchmark', 'compare', *records,
        '--output', COMPARISON_DIR / 'training-comparison.json', '--resume')
    source_only = Path(ROOT) / 'dev-source/dev-source.jsonl'
    run('dev-source', 'alignment_transfer_s2ut.prepare_inference_input', '--manifest', MANIFEST, '--split', 'dev', '--output', source_only, '--resume')
    if GENERATE_WAVEFORMS:
        run('vocoder', 'alignment_transfer_s2ut.fetch_assets', '--config', 'configs/s2ut/vocoder.yaml')
    evaluations = []
    table = ['| Condition | ASR-BLEU | UER |', '|---|---:|---:|']
    for name, label, record in conditions:
        predictions = Path(ROOT) / 'evaluation' / name / 'dev-predictions.jsonl'
        vocoder = ['--vocoder-config', 'configs/s2ut/vocoder.yaml'] if GENERATE_WAVEFORMS else ['--units-only']
        run(name + '-infer', 'alignment_transfer_s2ut.benchmark', 'infer', '--record', record, '--input', source_only,
            '--output', predictions, '--device', 'cuda', '--resume', *vocoder, gpu=True)
        metrics = predictions.with_name('dev-metrics.json')
        asr = ['--asr-config', 'configs/experiments/evaluation.yaml', '--device', 'cuda'] if ASR_BLEU else []
        run(name + '-evaluate', 'alignment_transfer_s2ut.evaluate', '--manifest', MANIFEST, '--split', 'dev',
            '--units', Path(ROOT) / 'units', '--predictions', predictions, '--output', metrics, '--resume', *asr, gpu=ASR_BLEU)
        evaluations.append(metrics)
        values = json.loads(metrics.read_text())
        bleu = f"{values['asr_bleu']:.2f}" if values.get('asr_bleu') is not None else '—'
        uer = f"{values['unit_error_rate']:.4f}" if values.get('unit_error_rate') is not None else '—'
        table.append(f'| {label} | {bleu} | {uer} |')
    run(comparison_name + '-comparison', 'alignment_transfer_s2ut.benchmark', 'compare', *records, '--evaluations', *evaluations,
        '--output', COMPARISON_DIR / 'comparison.json', '--resume')
    print(json.loads((COMPARISON_DIR / 'comparison.json').read_text()))
    display(Markdown('\n'.join(table)))
